In [ ]:
import os 
os.chdir(r'/Users/sachuriga/Desktop/code/nwb4fp/SRC')

from neurochat.nc_data import NData
from neurochat.nc_spike import NSpike
from neurochat.nc_spatial import NSpatial
import neurochat.nc_plot as nc_plot
from neurochat.nc_lfp import NLfp
import matplotlib.pyplot as plt
import numpy as np
from pynwb import NWBHDF5IO
import matplotlib.pyplot as plt
import numpy as np
import math
import pynapple as nap
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import normalize

import sys
import nwb4fp.analyses.maps as mapp
from nwb4fp.analyses.examples.tracking_plot import plot_ratemap,plot_path
from nwb4fp.analyses.fields import separate_fields_by_laplace, separate_fields_by_dilation,find_peaks,separate_fields_by_laplace_of_gaussian,calculate_field_centers,distance_to_edge_function, remove_fields_by_area, map_pass_to_unit_circle,which_field,compute_crossings
from elephant.statistics import time_histogram, instantaneous_rate
from nwb4fp.analyses import maps
from nwb4fp.analyses.data import pos2speed,speed_filtered_spikes,load_speed_fromNWB,load_units_fromNWB,get_filed_num,unit_location_ch
from scipy.ndimage import gaussian_filter
import ast
import pandas as pd
pd.set_option('display.max_rows', None)
np.set_printoptions(threshold=np.inf)
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

# 1. Load and Filter Data
df_loaded = pd.read_pickle(r'/Users/sachuriga/Desktop/Projects/CR_CA1_paper/tables/functional_properties_with_python_measurements_with_stability.pkl')
df_good = df_loaded[df_loaded['unit_quality'] == "good"]
df_a = df_good[df_good['session'] == "A"]

# 2. Define Groups and Constants
control_ids = ['65165', '65091', '63383', '66539', '65622']
exp_ids = ['65588', '63385', '66538', '66537', '66922']
control_color = 'blue'
exp_color = "red"

# Separate into control and experimental groups once (moved out of loop)
control_df = df_a[df_a['animal_id'].isin(control_ids)]
exp_df = df_a[df_a['animal_id'].isin(exp_ids)]

metrics = ['l_ratio', 'isi_violations_ratio', 'amplitude_median', 'snr']
titles = ['L ratio', 'ISI violations ratio', 'Amplitude median (mV)', 'Signal to noise ratio']
test_hy = ['two-sided', 'two-sided', 'two-sided', 'two-sided']

# 3. Setup Figure
fig = plt.figure(figsize=(7.2, 2.4), dpi=1200)
plt.rcParams.update({'font.size': 7,'font.family': 'sans-serif','font.sans-serif':['Arial', 'Calibri','DejaVu Sans', 'sans-serif']})
plt.rcParams.update({
    'axes.labelpad': 5,
    'ytick.major.pad': 2,
    'xtick.major.pad': 5,
    'ytick.major.size': 2,
    'xtick.major.size': 2
})

gs = gridspec.GridSpec(1, 4, height_ratios=[.8], width_ratios=[0.8, .8, .8, .8])
ax1_1 = fig.add_subplot(gs[0, 0])
ax1_2 = fig.add_subplot(gs[0, 1])
ax1_3 = fig.add_subplot(gs[0, 2])
ax1_4 = fig.add_subplot(gs[0, 3])

axes_list = [ax1_1, ax1_2, ax1_3, ax1_4]

# 4. Plotting Loop
for idx, ax in enumerate(axes_list):
    metric = metrics[idx]
    control_values = control_df[metric].dropna()
    exp_values = exp_df[metric].dropna()
    
    if len(control_values) > 0 and len(exp_values) > 0:
        # Stats printing
        control_mean = control_values.mean()
        exp_mean = exp_values.mean()
        control_sem = control_values.sem()
        exp_sem = exp_values.sem()
        print(f"\nComparison for {metric}:")
        print(f"Control mean: {control_mean:.2f} ± {control_sem:.2f}")
        print(f"Experimental mean: {exp_mean:.2f} ± {exp_sem:.2f}")
        
        # Mann-Whitney U test
        control_array = np.asarray(control_values.values, dtype=float)
        exp_array = np.asarray(exp_values.values, dtype=float)

        control_clean = control_array[~np.isnan(control_array)]
        exp_clean = exp_array[~np.isnan(exp_array)]

        u_stat, p_val = stats.mannwhitneyu(control_clean, exp_clean, alternative=test_hy[idx])

        # Prepare data for Seaborn plotting
        plot_df = pd.DataFrame({
            'value': pd.concat([control_values, exp_values]),
            'group': ['Control'] * len(control_values) + ['Experimental'] * len(exp_values)
        })
        
        # Filter out outliers (e.g., beyond 3 standard deviations)
        all_values = plot_df['value']
        mean_val = all_values.mean()
        std_val = all_values.std()
        plot_df_filtered = plot_df[(plot_df['value'] >= mean_val - 3 * std_val) & 
                                   (plot_df['value'] <= mean_val + 3 * std_val)]
        
        # Plotting
        sns.violinplot(
            data=plot_df_filtered, x='group', y='value', ax=ax, inner=None,
            palette={"Control": control_color, "Experimental": exp_color}, 
            width=0.8, cut=0, edgecolor='white'
        )
        sns.boxplot(
            data=plot_df_filtered, 
            x='group', y='value',
            palette={"Control": "black", "Experimental": "black"},
            width=0.3, 
            fill=False, 
            showfliers=False, 
            showmeans=False, 
            linewidth=1, 
            ax=ax 
        )

        # Formatting
        ax.set_ylabel(titles[idx])
        ax.set_xlabel('')
        ax.yaxis.grid(False)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['bottom'].set_visible(True)
        ax.spines['left'].set_visible(True)
        ax.set_xticklabels(['CR;DTA-', 'CR;DTA+'], rotation=-30)
        
        # Significance bars
        y_max = ax.get_ylim()[1]
        bar_height = y_max * 0.1 
        x_positions = [0, 1] 
        
        if (p_val < 0.05) & (p_val > 0.01):
            ax.plot([x_positions[0], x_positions[1]], [y_max + bar_height, y_max + bar_height], 
                    color='black', lw=1.5)
            ax.text(0.5, y_max + bar_height * 1.1, f'*', ha='center', va='bottom')
        elif (p_val < 0.01) & (p_val > 0.001):
            ax.plot([x_positions[0], x_positions[1]], [y_max + bar_height, y_max + bar_height], 
                    color='black', lw=1.5)
            ax.text(0.5, y_max + bar_height * 1.1, f'**', ha='center', va='bottom')
        elif p_val < 0.001:
            ax.plot([x_positions[0], x_positions[1]], [y_max + bar_height, y_max + bar_height], 
                    color='black', lw=1.5)
            ax.text(0.5, y_max + bar_height * 1.1, f'***', ha='center', va='bottom')

# 5. Save
fig.subplots_adjust(top=0.92, bottom=0.08, left=0.1, right=0.95, hspace=1.1, wspace=1.3)
plt.tight_layout()
plt.savefig(r'/Users/sachuriga/Desktop/Projects/CR_CA1_paper/Figures_mac/suppfig2.png', transparent=True, dpi=1200, bbox_inches='tight')